In [2]:
import joblib
import numpy as np
import dashscope
import json
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ========== 1. 设置 API Key 和加载模型 ==========
dashscope.api_key = ""

model = joblib.load("diabetes_model.pkl")
scaler = joblib.load("scaler.pkl")

# ========== 2. 定义8个指标的中文提示 ==========
feature_names = [
    "怀孕次数（没有就填0）",
    "血糖浓度（正常范围70-140）",
    "舒张压（正常范围60-90）",
    "三头肌皮褶厚度（正常范围10-50）",
    "2小时血清胰岛素（正常范围15-276）",
    "BMI（正常范围18.5-24.9）",
    "糖尿病遗传函数（0-2.5之间）",
    "年龄"
]

# ========== 3. 让用户逐个输入 ==========
print("=" * 40)
print("糖尿病风险预测小工具")
print("请依次输入以下指标：")
print("=" * 40)

patient_data = []
for name in feature_names:
    while True:
        try:
            value = float(input(f"{name}："))
            patient_data.append(value)
            break
        except ValueError:
            print("请输入数字！")

# ========== 4. 标准化 + 预测概率 ==========
patient_array = np.array([patient_data])   # 变成二维数组
patient_scaled = scaler.transform(patient_array)
risk_prob = model.predict_proba(patient_scaled)[0, 1]

print(f"\n模型预测患病概率：{risk_prob:.2%}")

# ========== 5. 调用 Qwen 输出 JSON ==========
patient_dict = dict(zip(feature_names, patient_data))

prompt = f"""
你是一名内分泌临床医师。请根据患者指标和模型预测概率，输出一段 JSON 格式的评估结果。
患者指标：{patient_dict}
模型预测患病概率：{risk_prob:.2%}

请严格按照以下 JSON 结构输出，不要包含任何其他文字：
{{
    "risk_level": "高风险/中风险/低风险",
    "main_factors": ["因素1", "因素2"],
    "suggestions": ["建议1", "建议2"],
    "disclaimer": "本结果仅供参考，不能替代临床就诊。"
}}
"""

resp = dashscope.Generation.call(
    model="qwen-max",
    messages=[{"role": "user", "content": prompt}]
)

# ========== 6. 解析并打印人类可读报告 ==========
if resp.status_code == 200:
    try:
        result = json.loads(resp.output.text)
        print("\n" + "=" * 40)
        print("      糖尿病风险评估报告")
        print("=" * 40)
        print(f"风险等级：{result['risk_level']}")
        print(f"主要因素：{'、'.join(result['main_factors'])}")
        print("生活建议：")
        for i, s in enumerate(result["suggestions"], 1):
            print(f"  {i}. {s}")
        print(f"\n免责声明：{result['disclaimer']}")
        print("=" * 40)
    except json.JSONDecodeError:
        print("解析失败！模型没有输出纯 JSON，请调整 Prompt。")
        print("原始输出：", resp.output.text)
else:
    print("调用失败：", resp.message)

糖尿病风险预测小工具
请依次输入以下指标：


怀孕次数（没有就填0）： 0
血糖浓度（正常范围70-140）： 96
舒张压（正常范围60-90）： 75
三头肌皮褶厚度（正常范围10-50）： 42
2小时血清胰岛素（正常范围15-276）： 150
BMI（正常范围18.5-24.9）： 25.5
糖尿病遗传函数（0-2.5之间）： 0
年龄： 19



模型预测患病概率：2.91%

      糖尿病风险评估报告
风险等级：低风险
主要因素：BMI略高、年龄较轻
生活建议：
  1. 保持健康饮食，控制体重在正常范围内。
  2. 定期进行身体检查，特别是关注血糖水平的变化。

免责声明：本结果仅供参考，不能替代临床就诊。
